In [3]:
from astropy.io import fits
import glob 
import matplotlib.pyplot as plt 
import numpy as np 
import pandas as pd
#from searchAdowload.utils import make_a_fits_list_csv,make_spectrum_fits
from astropy.table import Table
from sparcl.client import SparclClient
from astropy.io import fits
from dl import queryClient as qc
from astroquery.sdss import SDSS
from astropy.coordinates import SkyCoord
import astropy.units as u

import os
import time
from urllib.error import URLError

In [4]:
from astropy.io import fits
import glob 
import matplotlib.pyplot as plt 
import numpy as np 
import pandas as pd
#from searchAdowload.utils import make_a_fits_list_csv,make_spectrum_fits
from astropy.io import fits
from dl import queryClient as qc
from astroquery.sdss import SDSS
from astropy.table import Table

from sparcl.client import SparclClient
from astropy.coordinates import SkyCoord
import astropy.units as u
import os
from pathlib import Path
#from searchAdowload.dowload_rutines import desi_match,sdss_match
client = SparclClient(connect_timeout=3.1)

announcement=Data set deprecation notice: on November 19, 2025 the SDSS/BOSS DR16 data sets were deprecated. Please use the new SDSS/BOSS DR17 data sets instead.


In [ ]:
ra,dec = 184.0354010257332,0.6970763882738235

co = SkyCoord(ra=ra * u.deg, dec=dec * u.deg, frame="icrs")
data_release = 17  # SDSS DR
# radius must be an astropy quantity, e.g. radius = 2 * u.arcsec
radius = 1.8
rdeg = (radius * u.arcsec).to("deg")
jeje = SDSS.query_region(co,
			radius=rdeg,
			spectro=True,
			specobj_fields=[
				"specobjid", "plate", "mjd", "fiberID", "ra", "dec","run2d"
			],
			data_release=data_release,
		)

In [11]:
jeje

objID,specobjid,plate,mjd,fiberID,ra,dec,run2d
uint64,uint64,int64,int64,int64,float64,float64,str7
1237648722306269485,4330386778072832000,3846,55327,639,184.03538,0.69710199,v5_13_2


In [12]:
ra,dec = co.ra.value,co.dec.value
cons = {
	"ra": [ra - rdeg.value, ra + rdeg.value],
	"dec": [dec - rdeg.value, dec + rdeg.value],
	"data_release": ["DESI-DR1"],
}

outfields = ["sparcl_id", "specid", "ra", "dec", "redshift"]
found = client.find(
	outfields=outfields,
	constraints=cons,
	limit=20,  # get several, then choose nearest
)

In [13]:
found

Find Results: 2 records

In [ ]:
# from astroquery.ipac.ned import Ned

# Ned.TIMEOUT = 500  # increase from default 60 s
# refcode = "2024MNRAS.527.7055P"
# tab = Ned.query_refcode(refcode)

# df = tab.to_pandas()
# df.to_csv("2024MNRAS_527_7055P.csv",index=False)

inc = ["data_release", "datasetgroup", "dateobs", "dateobs_center", "dec",
	"exptime", "extra_files", "file", "flux", "instrument", "ivar", "mask",
	"model", "ra", "redshift", "redshift_err", "redshift_warning", "site",
	"sparcl_id", "specid", "specprimary", "spectype", "survey", "targetid",
	"telescope", "updated", "wave_sigma", "wavelength", "wavemax", "wavemin"]


def get_spectra_with_retry(best_match, data_release=17, n_try=5, wait=10):
    for i in range(n_try):
        try:
            return SDSS.get_spectra(matches=best_match, data_release=data_release)
        except Exception as e:
            print(f"Attempt {i+1}/{n_try} failed: {e}")
            if i < n_try - 1:
                time.sleep(wait)
            else:
                raise

def sdss_query_region_with_retry(co,radius, data_release=17, n_try=5, wait=10):
    from astropy.io.ascii.core import InconsistentTableError
    for i in range(n_try):
        try:
            return SDSS.query_region(co,
			radius=radius,
			spectro=True,
			specobj_fields=[
				"specobjid", "plate", "mjd", "fiberID", "ra", "dec","run2d"
			],
			data_release=data_release,
		)
        except Exception as e:
            if isinstance(e, InconsistentTableError):
                print(f"SDSS returned an invalid table: {type(e).__name__}: {e}")
                return []

            print(f"Attempt {i+1}/{n_try} failed: {type(e).__name__}: {e}")
            if i < n_try - 1:
                time.sleep(wait)
            else:
                return False
            
def sdss_match(fourmostid,co,radius,data_release = 17, n_try=5, wait=10):
    # matches = SDSS.query_region(co,
	# 		radius=radius,
	# 		spectro=True,
	# 		specobj_fields=[
	# 			"specobjid", "plate", "mjd", "fiberID", "ra", "dec","run2d"
	# 		],
	# 		data_release=data_release,
	# 	)
    # "run2d",
	#			"z", "zErr", "zWarning", "class", "subClass"
    matches = sdss_query_region_with_retry(co,radius, data_release=data_release, n_try=n_try, wait=wait)
    ra,dec = co.ra.value,co.dec.value
    if matches is None or len(matches) == 0:
        print(f"No SDSS spectrum within {radius} of ({ra}, {dec}) for target {fourmostid}")
        return False 
    else:
        if "ra" not in matches.colnames:
            print(f"No SDSS spectrum within {radius} of ({ra}, {dec}) for target {fourmostid}")
            return False #print()
        sep = co.separation(SkyCoord(matches["ra"] * u.deg, matches["dec"] * u.deg))
        best_idx = sep.argmin()
        best_match = matches[best_idx:best_idx + 1]

        #spectra = SDSS.get_spectra(matches=best_match, data_release=data_release)
        spectra = get_spectra_with_retry(
            best_match,
            data_release=data_release,
            n_try=n_try,
            wait=wait,
        )
        if spectra is None or len(spectra) == 0:
            print(f"SDSS match found but no spectrum retrieved for target {fourmostid}")
            return False 
        else:
            spec_hdul = spectra[0]
            hdr = spec_hdul[0].header
            hdr["OBJECT"] = str(fourmostid)
            hdr["CAT_RA"] = ra
            hdr["CAT_DEC"] = dec
            hdr["MATCH_RA"] = float(best_match["ra"][0])
            hdr["MATCH_DEC"] = float(best_match["dec"][0])
            hdr["SEP_ARC"] = float(sep[best_idx].arcsec)

            if "specobjid" in best_match.colnames:
                hdr["SPECOBJID"] = str(best_match["specobjid"][0])
            if "plate" in best_match.colnames:
                hdr["PLATE"] = int(best_match["plate"][0])
            if "mjd" in best_match.colnames:
                hdr["MJD"] = int(best_match["mjd"][0])
            if "fiberID" in best_match.colnames:
                hdr["FIBERID"] = int(best_match["fiberID"][0])
            if "z" in best_match.colnames:
                hdr["REDSHIFT"] = float(best_match["z"][0])
            if "zErr" in best_match.colnames:
                hdr["ZERR"] = float(best_match["zErr"][0])
            if "zWarning" in best_match.colnames:
                hdr["ZWARN"] = int(best_match["zWarning"][0])

            outfile = os.path.join("sdss_spectra_fits",f"spectrum_sdss_{str(fourmostid).replace(' ', '_')}.fits")
            spec_hdul.writeto(outfile, overwrite=True)
            #sdss_found = True
            print(f"Downloaded and saved SDSS spectrum for target {fourmostid} -> {outfile}")
            return True

def desi_match(fourmostid,co,radius,client=None):
    rdeg = radius.to(u.deg).value
    ra,dec = co.ra.value,co.dec.value
    cons = {
        "ra": [ra - rdeg, ra + rdeg],
        "dec": [dec - rdeg, dec + rdeg],
        "data_release": ["DESI-DR1"],
    }

    outfields = ["sparcl_id", "specid", "ra", "dec", "redshift"]
    found = client.find(
        outfields=outfields,
        constraints=cons,
        limit=20,  # get several, then choose nearest
    )

    if found.count == 0:
        print(f"No DESI spectrum within search box around ({ra}, {dec}) for target {fourmostid}")
        return False 
    else:
        # Choose nearest match on sky
        found_ra = [i["ra"] for i in found.records]
        found_dec = [i["dec"] for i in found.records]
        cand_coords = SkyCoord(np.array(found_ra)* u.deg, np.array(found_dec) * u.deg)
        
        cand_sep = co.separation(cand_coords)
        best_idx = cand_sep.argmin()

        if cand_sep[best_idx] > radius:
            print(f"DESI candidates found, but none within {radius} for target {fourmostid}")
            return False 
        else:
            sparcl_id = found.records[best_idx]["sparcl_id"]

            retrieved = client.retrieve(
                uuid_list=[sparcl_id],
                include=inc,
            )
            if len(retrieved.records) == 0:
                print(f"DESI match found but no spectrum retrieved for target {fourmostid}")
                return False 
            else:
                #retrieved.records[0]
                spec = retrieved.records[0]
                # header_meta = {
                #     'SPARCL_ID':      spec['sparcl_id'],
                #     'TARGETID':       spec['targetid'],
                #     'RA':             spec['ra'],
                #     'DEC':            spec['dec'],
                #     'REDSHIFT':       spec['redshift'],
                #     'SPECTYPE':       spec['spectype'],
                #     'REDSHIFT_ERR':   spec['redshift_err'],
                #     'REDSHIFT_WARN':  spec['redshift_warning'],
                #     'dateobs':         spec['dateobs'],
                #     "OBJECT": fourmostid
                #     # 'FLUX_R':         phot['flux_r'],
                #     # 'FLUX_Z':         phot['flux_z'],
                #     # 'MW_TRANS_G':     phot['mw_transmission_g'],
                #     # 'MW_TRANS_R':     phot['mw_transmission_r'],
                #     # 'MW_TRANS_Z':     phot['mw_transmission_z'],
                # }
                header_meta = {i:spec[i] for i in ["sparcl_id","targetid","ra","dec","redshift","spectype","redshift_err","redshift_warning","dateobs",
                                "data_release", "datasetgroup","dateobs_center",
                                "exptime", "instrument",
                                "specid", "specprimary", "survey",
                                "telescope", "updated", "wavemax", "wavemin"
                            ]}
                header_meta["OBJECT"] = fourmostid
                # # Write to FITS
                outfile = os.path.join("desi_spectra_fits", f"spectrum_desi_{str(fourmostid).replace(' ', '_')}.fits")
                make_spectrum_fits(outfile,spec['wavelength'], spec['flux'], spec['ivar'], spec['wave_sigma'],header_meta)
                print(f"Downloaded and saved DESI spectrum for target {fourmostid} -> {outfile} (SPARCL_ID={header_meta['sparcl_id']})")
                return True